In [1]:

# ── Colab setup ──────────────────────────────────────────────────────────────
# When running locally this cell does nothing.
# When running in Google Colab, clones the repo so all data files are available.
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists('raw_matches.csv'):
    REPO_URL = 'https://github.com/emaforlin/world_cup_learning_modernized.git'
    !git clone {REPO_URL} /content/wcl
    os.chdir('/content/wcl')
    sys.path.insert(0, '/content/wcl')


Cloning into '/content/wcl'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 42 (delta 17), reused 36 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 34.93 KiB | 8.73 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
import utils

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Normalization

In [3]:
input_cols = ['year',
              'matches_won_percent',
              'podium_score_yearly',
              'matches_won_percent_2',
              'podium_score_yearly_2',]
output_col = 'winner'

In [4]:
matches = utils.get_matches(with_team_stats=True, duplicate_with_reversed=True)
matches

,id,year,team1,score1,score2,team2,winner,matches_played,matches_won,years_played,...,podium_score_yearly,cups_won_yearly,matches_played_2,matches_won_2,years_played_2,podium_score_2,cups_won_2,matches_won_percent_2,podium_score_yearly_2,cups_won_yearly_2
0,0,1950,Brazil,4,0,Mexico,True,85.0,71.0,19.0,...,5.368421,0.263158,41.0,17.0,16.0,0.0,0.0,0.414634,0.000000,0.000000
1,1,1950,Yugoslavia,3,0,Switzerland,True,26.0,14.0,8.0,...,0.250000,0.000000,28.0,12.0,10.0,0.0,0.0,0.428571,0.000000,0.000000
3,3,1950,Yugoslavia,4,1,Mexico,True,26.0,14.0,8.0,...,0.250000,0.000000,41.0,17.0,16.0,0.0,0.0,0.414634,0.000000,0.000000
4,4,1950,Brazil,2,0,Yugoslavia,True,85.0,71.0,19.0,...,5.368421,0.263158,26.0,14.0,8.0,2.0,0.0,0.538462,0.250000,0.000000
5,5,1950,Switzerland,2,1,Mexico,True,28.0,12.0,10.0,...,0.000000,0.000000,41.0,17.0,16.0,0.0,0.0,0.414634,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1784,889,2022,Portugal,0,1,Morocco,False,29.0,17.0,8.0,...,0.750000,0.000000,16.0,5.0,6.0,2.0,0.0,0.312500,0.333333,0.000000
1785,890,2022,France,2,1,England,True,51.0,36.0,13.0,...,4.461538,0.153846,52.0,32.0,16.0,20.0,1.0,0.615385,1.250000,0.062500
1786,891,2022,Croatia,0,3,Argentina,False,22.0,13.0,6.0,...,2.666667,0.000000,62.0,41.0,16.0,56.0,3.0,0.661290,3.500000,0.187500
1787,892,2022,Morocco,0,2,France,False,16.0,5.0,6.0,...,0.333333,0.000000,51.0,36.0,13.0,58.0,2.0,0.705882,4.461538,0.153846


In [5]:
train, test = train_test_split(matches, test_size=0.2)

In [6]:
network = Sequential([
    Input((len(input_cols),)),
    Normalization(),
    Dense(10, activation='sigmoid'),
    Dense(10, activation='sigmoid'),
    Dense(1, activation='sigmoid'),
])

network.layers[0].adapt(train[input_cols].values)

network.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy',],
)


network.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization (Normalization)   │ (None, 5)              │            11 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           110 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 192 (772.00 B)

 Trainable params: 181 (724.00 B)

 Non-trainable params: 11 (48.00 B)

In [7]:
network.fit(
    train[input_cols], train[output_col],
    epochs=50,
    batch_size=128,
)

Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step - accuracy: 0.5036 - loss: 0.7110
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5036 - loss: 0.7041 
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5036 - loss: 0.6992 
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5036 - loss: 0.6955 
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5036 - loss: 0.6934 
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5036 - loss: 0.6913 
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5372 - loss: 0.6899 
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6924 - loss: 0.6882 
Epoch 9/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7223 - loss: 0.6866 
Epoch 10/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7223 - loss: 0.6850 
Epoch 11/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7187 - loss: 0.6832 
Epoch 12/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7205 - loss: 0.6814 


In [8]:
train_predictions = network.predict(train[input_cols])

accuracy_score(train[output_col], train_predictions.round())

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


0.7450090744101633

In [9]:
test_predictions = network.predict(test[input_cols])

accuracy_score(test[output_col], test_predictions.round())

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


0.7572463768115942

In [10]:
def predict(year, team1, team2):
    case_inputs = utils.build_inputs_for_match(year, team1, team2, input_cols)
    result = network.predict(case_inputs)

    if result > 0.5:
        winner = team1
    else:
        winner = team2

    print(result[0][0], '→', winner)

In [11]:
predict(1950, 'Mexico', 'Brazil')  # real result: 4-0 wins Brazil

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
0.22393711 → Brazil


In [12]:
predict(1990, 'United Arab Emirates', 'Colombia')  # real result: 2-0 wins Colombia

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
0.2675819 → Colombia


In [13]:
predict(2002, 'South Africa', 'Spain')  # real result: 2-3 wins Spain

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
0.3111926 → Spain


In [14]:
predict(2010, 'Japan', 'Cameroon')  # real result: 1-0 wins Japan

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
0.53619486 → Japan


In [15]:
predict(2014, 'Argentina', 'Germany')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
0.3771652 → Germany
